# Init

In [7]:
# init
from importlib import reload
import os
from pathlib import Path
import pandas as pd
from IPython.display import clear_output
import boto3
import sys

import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph
import toolsSync.main as tsm

def pckgs_reload():
    reload(tgm)
    reload(too)
    reload(tph)
    reload(tgl)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
TESTS_DIR = DATA_DIR / 'tests results'
CLEANED_DIR = DATA_DIR / 'cleaned'
DEV_MODE = False

TEST_BASIC_DIR = TESTS_DIR / 'osm basic test'
process_state_file = DATA_DIR / "process_state.json"
process_state = tgm.load(process_state_file)
print(len(process_state))
basic_test_to_delete_file = TEST_BASIC_DIR / "basic_test_to_delete.json"
logger = tgl.initiate_logger('logger', TEST_BASIC_DIR / 'basic_test.log')

214


## * import

In [9]:
data_dir = DATA_DIR / 'cleaned'
files = [f for f in data_dir.rglob('*.pkl')]
print(len(files))

df_by_cntr = {file.parent.name:tgm.load(str(file)) for file in files}
print(len(df_by_cntr))

83
83


In [10]:
df_all = pd.concat(df_by_cntr.values(), ignore_index=True)
print(len(df_all))

155843


In [3]:
id_tuples = df_all[['id','tags.parent_id','tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
print(len(id_tuples))

id_tuples_df = pd.DataFrame(id_tuples, columns=['id','tags.parent_id','tags.country_id'])
print(len(id_tuples_df))

155100
155100


### * select to test

In [151]:
countries_tested = [c for c, val in process_state.items() if (val['test_basic']['status'] == 'ok')]
logger.info(f"countries tested: {len(countries_tested)}")
countries_to_test = [c for c, val in process_state.items() if 
    (val['clean']['status'] == 'ok') and (val['test_basic']['status'] in ['pending', 'error'])
]
logger.info(f"countries to test: {len(countries_to_test)}")

[INFO] : countries tested: 83
[INFO] : countries to test: 0


### * initialize B2

In [126]:
session = boto3.session.Session()

s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

### * download required data

In [4]:
logger.info(f"* Downloading required data to test: {len(countries_to_test)} countries")
logger.info(f"  * Downloading country data from B2 in directory: '{CLEANED_DIR.relative_to(ROOT)}'")

countries_downloaded = tsm.donwload_country_data_from_bucket(countries_to_test, os.environ["B2_BUCKET_NAME"], CLEANED_DIR.relative_to(ROOT), CLEANED_DIR, s3, logger)

logger.info(f'* Countries to test: {len(countries_to_test)}')
logger.info(f"* Countries to test with downloaded data from B2: {len(countries_downloaded)}")

[INFO] : * Downloading required data to test: 11 countries
[INFO] :   * Downloading country data from B2 in directory: 'data\cleaned'
[INFO] :   * Total objects found in B2 'data\cleaned': 1
[INFO] :   * Total countries found in B2 'data\cleaned': 1
[INFO] :   * Countries to download: 11: ['Benin', 'Bolivia', 'BritishVirginIslands', 'Belize', 'Azerbaijan', 'Bahrain', 'Bahamas', 'Barbados', 'Botswana', 'Bermuda', 'BosniaAndHerzegovina']
[INFO] :   * Countries to download found in B2: 0
[INFO] :   * Countries to download missing in B2: 0
[INFO] :   * Downloading data for countries: 0
[INFO] :   * Downloading directory: 'data\cleaned' -> 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned'
[INFO] :     * (1/11) Country Benin files found: 0
[INFO] :     * (2/11) Country Bolivia files found: 0
[INFO] :     * (3/11) Country BritishVirginIslands files found: 0
[INFO] :     * (4/11) Country Belize files found: 0
[INFO] :     * (5/11) Country Azerbaijan 

### * load data for countries to test

In [134]:
logger.info(f"* Load data from: {CLEANED_DIR.relative_to(ROOT)}")
cleaned_files = [f for f in CLEANED_DIR.glob('*/*')]
logger.info(f"  * Directories found: {len(cleaned_files)}")

# to_test_df = {file.parent.name:tgm.load(str(file)) for file in cleaned_files if file.parent.name}
to_test_df = {file.parent.name:tgm.load(str(file)) for file in cleaned_files if file.parent.name in countries_to_test}
logger.info(f"  * Countries to test: {len(countries_to_test)}")
logger.info(f"  * Data loaded for countries to test: {len(to_test_df)}")

[INFO] : * Load data from: data\cleaned
[INFO] :   * Directories found: 83
[INFO] :   * Countries to test: 11
[INFO] :   * Data loaded for countries to test: 11


### * make test

In [135]:
logger.info(f"* Make basic test and save")
for country, df in to_test_df.items():
    logger.info(f'* Country: {country}')
    test_res = too.osm_basic_test(df)
    tgm.dump(TEST_BASIC_DIR / country / f'{country}_basic_test_res.pkl', test_res)

[INFO] : * Make basic test and save
[INFO] : * Country: Azerbaijan
[INFO] :  * missing names: 0
[INFO] :  * relations from other countries: 0
[INFO] : * Country: Bahamas
[INFO] :  * missing names: 0
[INFO] :  * relations from other countries: 0
[INFO] : * Country: Bahrain
[INFO] :  * missing names: 0
[INFO] :  * relations from other countries: 0
[INFO] : * Country: Barbados
[INFO] :  * missing names: 0
[INFO] :  * relations from other countries: 0
[INFO] : * Country: Belize
[INFO] :  * missing names: 0
[INFO] :  * relations from other countries: 0
[INFO] : * Country: Benin
[INFO] :  * missing names: 0
[INFO] :  * relations from other countries: 0
[INFO] : * Country: Bermuda
[INFO] :  * missing names: 0
[INFO] :  * relations from other countries: 0
[INFO] : * Country: Bolivia
[INFO] :  * missing names: 1
[INFO] :  * relations from other countries: 1
[INFO] : * Country: BosniaAndHerzegovina
[INFO] :  * missing names: 0
[INFO] :  * relations from other countries: 0
[INFO] : * Country: Bot

### * review

In [91]:
test_res_by_cntr = {f.parent.name:tgm.load(f) for f in TEST_BASIC_DIR.glob('*/*')}
logger.info(f'Test results found: {len(test_res_by_cntr)}')

[INFO] : Test results found: 83


In [92]:
files = [f for f in CLEANED_DIR.rglob('*.pkl')]
df_by_cntr = {file.parent.name:tgm.load(str(file)) for file in files}
logger.info(f"Data loaded for countries to test: {len(df_by_cntr)}")

[INFO] : Data loaded for countries to test: 83


In [93]:
temp = pd.DataFrame(test_res_by_cntr)
temp = temp.T.map(len)
temp['test_tags_total'] = [len(elems) for c,elems in df_by_cntr.items() if c in test_res_by_cntr.keys()]
temp.peek()

,missing_name,test_tags_leak,test_tags_in_country,test_tags_NA_result,test_tags_total
Afghanistan,0,0,24,11,35
Albania,0,0,390,0,390
Algeria,0,0,1,2147,2148
Andorra,0,0,1,0,1
Angola,0,0,1,63,64
Anguilla,0,0,1,0,1
AntiguaAndBarbuda,0,0,1,11,12
Argentina,1,0,885,326,1211
Armenia,0,0,27,0,27
Australia,0,0,1,615,616


### * select missing names and leaks from other countries

In [136]:
test_res_by_cntr = {f.parent.name:tgm.load(f) for f in TEST_BASIC_DIR.glob('*/*') if f.parent.name in countries_to_test}
# test_res_by_cntr = {f.parent.name:tgm.load(f) for f in TEST_BASIC_DIR.glob('*/*') if f.parent.name}
logger.info(f'Test results found: {len(test_res_by_cntr)}')

[INFO] : Test results found: 11


In [137]:
missing_names = set()
leaks = set()
for k,v in test_res_by_cntr.items():
    missing_names.update(v['missing_name'])
for k,v in test_res_by_cntr.items():
    leaks.update(v['test_tags_leak'])

logger.info(f"Missing names relations: {len(missing_names)}")
logger.info(f"Leaks relations: {len(leaks)}")
relations_from_test_to_delete = leaks | missing_names
logger.info(f"To delete parents relations: {len(relations_from_test_to_delete)}")

[INFO] : Missing names relations: 1
[INFO] : Leaks relations: 1
[INFO] : To delete parents relations: 2


### * From the relations to delete, select the childs in the country that has them as parent

In [138]:
id_triplets = pd.concat(to_test_df.values(), ignore_index=True)[['id', 'tags.parent_id', 'tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
logger.info(f"All dataframes id triplets: {len(id_triplets)}")

[INFO] : All dataframes id triplets: 743


In [139]:
parents_to_delete = relations_from_test_to_delete
relations_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    childs_to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    relations_childs_to_delete.update(childs_to_delete)
    parents_to_delete = childs_to_delete
logger.info(f"Childs to delete: {len(relations_childs_to_delete)}")

[INFO] : Childs to delete: 4


### * Add to basic_test_to_delete.json and save

In [140]:
basic_test_to_delete = relations_from_test_to_delete | relations_childs_to_delete
logger.info(f"Basic test to delete relations: {len(basic_test_to_delete)}")

[INFO] : Basic test to delete relations: 6


In [ ]:
basic_test_to_delete_old = tgm.load(TEST_BASIC_DIR / "basic_test_to_delete.json")
basic_test_to_delete_new = basic_test_to_delete_old | basic_test_to_delete
logger.info(f"Current total of relations to delete from basic test: {len(basic_test_to_delete_new)}")

[INFO] : Current total of relations to delete from basic test: 4199


In [ ]:
tgm.dump(TEST_BASIC_DIR / "basic_test_to_delete.pkl", basic_test_to_delete_new)

### * upload results to B2 and update process_state

In [ ]:
logger.info("* Uploading data to backblaze b2")
config = {'root':ROOT, 's3':s3, 'logger':logger}
task = 'test_basic'
tested_dirs = [dir.name for dir in TEST_BASIC_DIR.glob('*/') if dir.name in countries_to_test]

if len(tested_dirs) < 1:
    logger.info("No data to upload, exiting script")
    sys.exit(0)
else:
    logger.info(f"* Test result directories found: {len(tested_dirs)}")

for country in tested_dirs:
    process_status = 'ok'
    process_error = None
    
    # all data in country directory will  be uploaded
    if not DEV_MODE:
        upload_response = tsm.upload_dir_files_to_backblaze(TEST_BASIC_DIR / country, config)
        process_status = upload_response['status']
        process_error = upload_response['status_type']

    # override process task state with upload response
    logger.info(f"  * Updating {country} in process state: ({task}, ok)")
    tsm.update_process_state(process_state, country, task, process_status=process_status, process_error=process_error)


[INFO] : * Uploading data to backblaze b2
[INFO] : * Test result directories found: 11
[INFO] :   * Updating Azerbaijan in process state: (test_basic, ok)
[INFO] :   * Updating Bahamas in process state: (test_basic, ok)
[INFO] :   * Updating Bahrain in process state: (test_basic, ok)
[INFO] :   * Updating Barbados in process state: (test_basic, ok)
[INFO] :   * Updating Belize in process state: (test_basic, ok)
[INFO] :   * Updating Benin in process state: (test_basic, ok)
[INFO] :   * Updating Bermuda in process state: (test_basic, ok)
[INFO] :   * Updating Bolivia in process state: (test_basic, ok)
[INFO] :   * Updating BosniaAndHerzegovina in process state: (test_basic, ok)
[INFO] :   * Updating Botswana in process state: (test_basic, ok)
[INFO] :   * Updating BritishVirginIslands in process state: (test_basic, ok)


### commit updates

In [ ]:
tgm.dump(basic_test_to_delete_file, basic_test_to_delete_new)
tgm.dump(process_state_file, process_state)
if not DEV_MODE:
    tsm.commit_file(process_state_file, f"Update process state for {country}: ({task}, ok)", config['logger'])
    tsm.commit_file(basic_test_to_delete_file, f"Update basic_test_to_delete relations, current length {len(basic_test_to_delete_new)}", logger)


## 3. osm duplicates test

### * check data

In [14]:
dups_id = df_all[df_all.duplicated(subset=['id'])]['id'].to_list()
print('unique duplicates (id):', len(dups_id))

dups_df = df_all[df_all.duplicated(subset=['id'], keep=False)]

dups_id_tuples = dups_df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).to_list()
print('rows with duplicates (id):', len(dups_id_tuples))

unique duplicates (id): 3454
rows with duplicates (id): 5979


In [5]:
print('rows with duplicates (id):', len(id_tuples_df[id_tuples_df['id'].duplicated(keep=False)]))
print('unique duplicates (id):', len(id_tuples_df[id_tuples_df['id'].duplicated()]))

rows with duplicates (id): 5979
unique duplicates (id): 3454


In [ ]:
print(
    'rows with duplicates (id, parent):',
    len(id_tuples_df[id_tuples_df[['id','tags.parent_id']].duplicated(keep=False)])
)
# zero result here means that all rows are unique and the duplicates comes from entities that are incorrectly listed as child
print(
    'rows with duplicates (id, parent_id, country_id):',
    len(id_tuples_df[id_tuples_df[['id','tags.parent_id','tags.country_id']].duplicated(keep=False)])
)

rows with duplicates (id, parent): 1418
rows with duplicates (id, parent_id, country_id): 0


In [7]:
to_review_ids = [('72639','60189','60189'), ('72639','60199','60199'),('6543282', '6543265', '192756'),
 ('6543282', '6543273', '192756'), ('59190', '59195', '59065'), ('59190', '59752', '59065')]

In [8]:
tgm.dump(TEST_RESULTS_DIR / 'osm duplicates test' / "to_review_ids.pkl", to_review_ids)

### [dev]

In [ ]:
temp1 = too.is_child_inside_parent('6543282','6543265')
[[k,v['result']] for k,v in temp1.items()]

[INFO] :    > Getting admin_centre:
[INFO] :     * Found node (admin_centre)
[INFO] :    > Testing admin_centre node (id: 7934379969)
[INFO] :     * Finished testing (admin_centre): False
[INFO] :    > Getting label:
[INFO] :     * Missing node (label)
[INFO] :    > Getting place:
[INFO] :     * Found node (place)
[INFO] :    > Testing place node (id: 7934379969)
[INFO] :     * Finished testing (place): False
[INFO] :    > Getting geom_node:
[INFO] :     * Found node (geom_node)
[INFO] :    > Testing geom_node node (id: 4298631214)
[INFO] :     * Finished testing (geom_node): True
[INFO] :    > Getting centroid:
[INFO] :     * Found node (centroid)
[INFO] :    > Testing centroid (lat: 36.8944071, lon: 6.4872707)
[INFO] :     * Finished testing (centroid): True


[['admin_centre', False],
 ['label', None],
 ['place', False],
 ['geom_node', True],
 ['centroid', True]]

In [ ]:
tph.get_from_df(df_all,['id','tags.parent_id','tags.country_id'],('6543282', '6543265', '192756'))

,type,id,tags.admin_level,tags.parent_id,tags.name,tags.name:us,tags.ISO3166-1,tags.ISO3166-2,tags.is_in:country,tags.ref:nuts,tags.ref:nuts:2,tags.ref:nuts:3,tags.addr:country,tags.country_name,tags.country_id
2462,rel,6543282,8,6543265,Beni Zid ⴱⴻⵏⵉ ⵣⵉⴷ بنى زيد,<NA>,<NA>,<NA>,Algeria,<NA>,<NA>,<NA>,<NA>,Algeria,192756


### * select to test

In [19]:
dups_test_processed = tgm.load(TEST_RESULTS_DIR / 'osm duplicates test' / "dups_test_processed.pkl")
print(len(dups_test_processed))

1819


In [20]:
first_level_tuples_to_delete = tgm.load(TEST_RESULTS_DIR / 'osm first level test'/ "tuples_to_delete.pkl")
print(len(first_level_tuples_to_delete))

4932


In [21]:
to_process = {}
# there are duplicates between countries, so checking by country is not enough. Use list of all duplicates by id found
print('id duplicates count:', len(dups_id_tuples))
print('id tuples processed:', len(dups_test_processed))

for country, df in df_by_cntr.items():
    df_id_tuples = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1)
    to_process_bool_series = (
        df_id_tuples.isin(dups_id_tuples) & 
        ~df_id_tuples.isin(dups_test_processed) & 
        ~df_id_tuples.isin(first_level_tuples_to_delete)
    )
    
    to_process_df = df[to_process_bool_series]
    if not to_process_df.empty:
        to_process[country] = to_process_df

to_process_all_tuples = [ele for df in to_process.values() for ele in df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1)]

print('countries to process: ', len(to_process))
print('id tuples to process: ', len(to_process_all_tuples))

id duplicates count: 5979
id tuples processed: 1819
countries to process:  1
id tuples to process:  3744


### * make tests

In [22]:
logger = tgl.initiate_logger('logger')
save_logger_path = TEST_RESULTS_DIR / 'osm duplicates test' / 'dups_test_res.log'
tgl.add_file_handler(logger, save_logger_path)

In [24]:
for country, df in to_process.items():
    save_path = TEST_RESULTS_DIR / 'osm duplicates test' / country / f"{country}_dups_test_res.pkl"
    test_res_stored = tgm.load(save_path) if os.path.exists(save_path) else {}
    processed_tuples = test_res_stored.keys()

    to_test_df = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(processed_tuples)
    to_test_df = df[~to_test_df]

    test_res = too.osm_test_center(
        to_test_df,
        save_temp=True,
        save_path=save_path
    )
    clear_output(wait=True)

[INFO] : * country: UnitedStates
[INFO] :  ^ [1/3744]: testing ('2219688', '1116270', '148838'):
[INFO] :    > Getting admin_centre:
[INFO] :     * Found node (admin_centre)
[INFO] :    > Testing admin_centre node (id: 5345974620)
[INFO] :     * Finished testing (admin_centre): False
[INFO] :    > Getting label:
[INFO] :     * Found node (label)
[INFO] :    > Testing label node (id: 6605516560)
[INFO] :     * Finished testing (label): False
[INFO] :    > Getting place:
[INFO] :     * Found node (place)
[INFO] :    > Testing place node (id: 6605516560)
[INFO] :     * Finished testing (place): False
[INFO] :    > Getting geom_node:
[INFO] :     * Found node (geom_node)
[INFO] :    > Testing geom_node node (id: 1779620941)
[INFO] :    * Attempt 1 failed: http_error: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
[INFO] :     * Finished testing (geom_node): False
[INFO] :    > Getting centroid:
[INFO]

KeyboardInterrupt: 

In [ ]:
('5320537', '9583346', '286393')

### * review

In [59]:
files = [f for f in (TEST_RESULTS_DIR / 'osm duplicates test').glob('*/*')]
print(len(files))
test_res_by_cntr = {}
for f in files:
    test_res_by_cntr[f.parent.name] = tgm.load(str(f))

print(len(test_res_by_cntr))

test_res_by_elem = {k:v for list in test_res_by_cntr.values() for k,v in list.items()}
print(len(test_res_by_elem))

30
30
1819


In [60]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1480
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               154
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                         151
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               26
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [61]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         6980
missing    2115
Name: count, dtype: int64

### * update processed data

In [62]:
dups_test_processed = [ele for list in list(test_res_by_cntr.values()) for ele in list]
print(len(dups_test_processed))

1819


In [63]:
tgm.dump((TEST_RESULTS_DIR / 'osm duplicates test' / "dups_test_processed.pkl"), dups_test_processed)

### * review

In [ ]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1399
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               154
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                          55
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               20
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [ ]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         6242
missing    1938
Name: count, dtype: int64

### * select false entities and childs to delete

In [30]:
test_res_simplified = {}
for id, val in test_res_by_elem.items():
    true_count = 0
    false_count = 0
    for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
        if val[type]['status'] == 'ok':
            # select the first one that gives True
            # test_res_simplified[id] = False
            # if val[type]['result'] is True:
            #     test_res_simplified[id] = val[type]['result']
            #     break

            # assign the first valid result
            # test_res_simplified[id] = val[type]['result']
            # break

            # make a voting weight selection
            if val[type]['result'] is True: true_count += 1
            else: false_count += 1
    
    test_res_simplified[id] = true_count >= false_count


pd.Series([v for k,v in test_res_simplified.items()]).value_counts()

True     1616
False     189
Name: count, dtype: int64

In [31]:
dups_tuples_to_delete_parents = [k for k,v in test_res_simplified.items() if v is False]
print(len(dups_tuples_to_delete_parents))

189


In [33]:
temp = [[ele[0],ele[2]] for ele in dups_tuples_to_delete_parents]
dups_tuples_to_delete_childs = [ele for ele in id_tuples if [ele[1],ele[2]] in temp]
print(len(dups_tuples_to_delete_childs))

# dups_tuples_to_delete = dups_tuples_to_delete_childs + dups_tuples_to_delete_parents
dups_tuples_to_delete = dups_tuples_to_delete_parents
print(len(dups_tuples_to_delete))

930
189


In [ ]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

### * add manual review deletion

In [35]:
to_review_ids = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "to_review_ids.pkl"))
print(len(to_review_ids))

6


In [39]:
dups_tuples_to_delete_manual = [('6543282', '6543273', '192756'), ('59190', '59195', '59065')]

In [40]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete_manual.pkl"), dups_tuples_to_delete_manual)

In [41]:
dups_tuples_to_delete = dups_tuples_to_delete + dups_tuples_to_delete_manual
print(len(dups_tuples_to_delete))

191


In [42]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

In [43]:
print(len(dups_id_tuples))
print(len(dups_tuples_to_delete))
temp = tgm.complement(dups_id_tuples, dups_tuples_to_delete)
print(len(temp))

1805
191
1614
